# YouTube Caption Fetcher — Indian ASR Benchmark

**Stage 1 · Task 4 · Google Colab edition**

Fetches English captions (manual → auto fallback) for all 986 samples in the `raianand/TIE_shorts` test split.  
Progress is checkpointed to Google Drive **after every row** — safe to interrupt and resume.

### Workflow
1. Run **Cell 1** once per session (mounts Drive, installs deps)
2. Run **Cell 2** to set config
3. Run **Cell 3** to load the dataset
4. Run **Cell 4** to load resume state
5. Run **Cell 5** to fetch captions — re-run after any interruption to resume
6. Run **Cell 6** when all rows are done to write the final CSV

### After completion
Download `wer_youtube_raw.csv` from `My Drive/indian-asr-bench/` and place it in  
`results/stage1_raw_transcripts/wer_youtube_raw.csv` in the repo, then run:  
```bash
python normalize_and_score.py
```

In [1]:
# ── Cell 1: Setup ────────────────────────────────────────────────────────────
# Run once per Colab session.

from google.colab import drive
drive.mount('/content/drive')

!pip install -q youtube-transcript-api datasets pandas tqdm

import os
os.makedirs('/content/drive/MyDrive/indian-asr-bench', exist_ok=True)
print('Drive mounted. Folder ready: My Drive/indian-asr-bench/')

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 485.2/485.2 kB 31.3 MB/s eta 0:00:00
Drive mounted. Folder ready: My Drive/indian-asr-bench/


In [2]:
# ── Cell 2: Config ───────────────────────────────────────────────────────────
# Adjust these values if needed.

import os

DRIVE_DIR       = '/content/drive/MyDrive/indian-asr-bench'
CHECKPOINT_PATH = f'{DRIVE_DIR}/wer_youtube_checkpoint.csv'
FINAL_PATH      = f'{DRIVE_DIR}/wer_youtube_raw.csv'
HF_CACHE_DIR    = f'{DRIVE_DIR}/hf_cache'  # dataset cached on Drive — survives session restarts

DELAY_BETWEEN_REQUESTS = 1.5   # seconds between successful requests
IP_BLOCK_WAIT          = 90    # seconds to wait when IP-blocked before retrying
MAX_IP_BLOCKS          = 3     # stop after this many consecutive IP blocks
RETRY_ON_ERROR         = 2     # retries per video on non-block errors
RETRY_DELAY            = 3     # seconds between retries

os.makedirs(HF_CACHE_DIR, exist_ok=True)

print('Config loaded.')
print(f'  Checkpoint : {CHECKPOINT_PATH}')
print(f'  Final CSV  : {FINAL_PATH}')
print(f'  HF cache   : {HF_CACHE_DIR}')
print(f'  Delay      : {DELAY_BETWEEN_REQUESTS}s/request')

Config loaded.
  Checkpoint : /content/drive/MyDrive/indian-asr-bench/wer_youtube_checkpoint.csv
  Final CSV  : /content/drive/MyDrive/indian-asr-bench/wer_youtube_raw.csv
  HF cache   : /content/drive/MyDrive/indian-asr-bench/hf_cache
  Delay      : 1.5s/request


In [3]:
# ── Cell 3: Load Dataset ─────────────────────────────────────────────────────
# Dataset is cached to Drive on first run (~200 MB).
# Subsequent runs (new sessions) load from Drive cache — no re-download.

import os
import collections
from datasets import load_dataset

os.environ['HF_DATASETS_CACHE'] = HF_CACHE_DIR

cache_exists = os.path.isdir(HF_CACHE_DIR) and any(
    f.startswith('raianand') for f in os.listdir(HF_CACHE_DIR)
)
print(f'HF cache dir  : {HF_CACHE_DIR}')
print(f'Cache present : {cache_exists} {"(will load from Drive)" if cache_exists else "(will download once ~200 MB)"}')
print('Loading raianand/TIE_shorts (test split) ...')

dataset = load_dataset('raianand/TIE_shorts', split='test', cache_dir=HF_CACHE_DIR)
print(f'Loaded {len(dataset)} samples.')

regions = collections.Counter(dataset['Native_Region'])
classes = collections.Counter(dataset['Speech_Class'])
genders = collections.Counter(dataset['Gender'])
print(f'  Regions : {dict(regions)}')
print(f'  Classes : {dict(classes)}')
print(f'  Genders : {dict(genders)}')

HF cache dir  : /content/drive/MyDrive/indian-asr-bench/hf_cache
Cache present : True (will load from Drive)
Loading raianand/TIE_shorts (test split) ...


README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/26 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/26 [00:00<?, ?it/s]

Loaded 986 samples.
  Regions : {'EAST': 352, 'SOUTH': 363, 'NORTH': 202, 'WEST': 69}
  Classes : {'SLOW': 374, 'FAST': 413, 'AVG': 199}
  Genders : {'M': 928, 'F': 58}


In [4]:
# ── Cell 4: Load Resume State ────────────────────────────────────────────────
# Re-run this cell after any interruption before re-running Cell 5.

import os
import pandas as pd

completed_ids: set = set()

def _load_ids_from_csv(path: str) -> set:
    df = pd.read_csv(path)
    return set(df['ID'].astype(str).tolist())

# Always populate completed_ids from whatever exists — checkpoint takes priority
if os.path.exists(CHECKPOINT_PATH):
    completed_ids = _load_ids_from_csv(CHECKPOINT_PATH)
    df_ckpt = pd.read_csv(CHECKPOINT_PATH)
    remaining = len(dataset) - len(completed_ids)
    print(f'[RESUME] Checkpoint found: {len(completed_ids)} done, {remaining} remaining.')
    if 'caption_type' in df_ckpt.columns:
        print(f'  Type breakdown: {dict(df_ckpt["caption_type"].value_counts())}')
elif os.path.exists(FINAL_PATH):
    # Final CSV exists — load its IDs so Cell 5 skips already-fetched rows
    completed_ids = _load_ids_from_csv(FINAL_PATH)
    remaining = len(dataset) - len(completed_ids)
    print(f'[PARTIAL FINAL] wer_youtube_raw.csv exists with {len(completed_ids)} rows, {remaining} remaining.')
    print('  Cell 5 will skip these and continue fetching the rest.')
    print('  (If run is truly complete, skip to Cell 6.)')
else:
    print(f'[FRESH] No checkpoint found. Starting from scratch ({len(dataset)} samples).')

checkpoint_file_exists = os.path.exists(CHECKPOINT_PATH)
print(f'\nReady. {len(completed_ids)} IDs will be skipped in Cell 5.')

[RESUME] Checkpoint found: 912 done, 74 remaining.
  Type breakdown: {'unavailable': np.int64(850), 'manual': np.int64(191), 'error': np.int64(10), 'ip_blocked': np.int64(1)}

Ready. 912 IDs will be skipped in Cell 5.


In [5]:
# ── Cell 5: Fetch Captions ───────────────────────────────────────────────────
# Re-run this cell to resume after any interruption.
# Progress is written to Drive after EVERY row.

import os
import time
import pandas as pd
from tqdm.notebook import tqdm
from youtube_transcript_api import YouTubeTranscriptApi

# If resuming from final CSV (no checkpoint), seed checkpoint from it first
# so append logic has a consistent base to build on
if not os.path.exists(CHECKPOINT_PATH) and os.path.exists(FINAL_PATH):
    pd.read_csv(FINAL_PATH).to_csv(CHECKPOINT_PATH, index=False)
    print(f'Seeded checkpoint from existing final CSV ({len(completed_ids)} rows).')

api = YouTubeTranscriptApi()
consecutive_ip_blocks = 0
ip_block_stop = False

stats = {'manual': 0, 'auto': 0, 'unavailable': 0, 'ip_blocked': 0, 'error': 0}

# Seed stats from checkpoint
if os.path.exists(CHECKPOINT_PATH):
    _df = pd.read_csv(CHECKPOINT_PATH)
    if 'caption_type' in _df.columns:
        for _t, _c in _df['caption_type'].value_counts().items():
            stats[str(_t)] = int(_c)

_IP_BLOCK_SIGNALS = ('IpBlocked', 'RequestBlocked', 'TooManyRequests', 'ip_blocked', 'too many requests', 'blocked')

def _is_ip_block(err_type: str, err_msg: str) -> bool:
    combined = (err_type + ' ' + err_msg).lower()
    return any(s.lower() in combined for s in _IP_BLOCK_SIGNALS)

def _is_unavailable(err_type: str) -> bool:
    return any(x in err_type for x in [
        'TranscriptsDisabled', 'NoTranscriptFound',
        'NoTranscriptAvailable', 'VideoUnavailable',
        'NotTranslatable', 'CouldNotRetrieveTranscript',
    ])


def fetch_caption(video_id: str) -> tuple:
    """Fetch English captions. Returns (text, caption_type).

    Priority: manual English → auto-generated English.
    caption_type values: 'manual', 'auto', 'unavailable', 'ip_blocked', 'error'
    """
    global consecutive_ip_blocks

    for attempt in range(RETRY_ON_ERROR + 1):
        try:
            transcript_list = api.list(video_id)

            fetched_text = None
            fetched_kind = None

            try:
                t = transcript_list.find_manually_created_transcript(['en'])
                data = t.fetch()
                fetched_text = ' '.join(s.text for s in data).strip()
                fetched_kind = 'manual'
            except Exception:
                pass

            if fetched_text is None:
                try:
                    t = transcript_list.find_generated_transcript(['en'])
                    data = t.fetch()
                    fetched_text = ' '.join(s.text for s in data).strip()
                    fetched_kind = 'auto'
                except Exception:
                    pass

            if fetched_text is not None:
                consecutive_ip_blocks = 0
                return fetched_text, fetched_kind
            return '', 'unavailable'

        except Exception as e:
            err_type = type(e).__name__
            err_msg  = str(e)

            if _is_ip_block(err_type, err_msg):
                consecutive_ip_blocks += 1
                action = 'retrying' if consecutive_ip_blocks < MAX_IP_BLOCKS else 'stopping'
                tqdm.write(
                    f'  [IP BLOCKED] #{consecutive_ip_blocks}/{MAX_IP_BLOCKS} ({err_type}) — '
                    f'waiting {IP_BLOCK_WAIT}s then {action}...'
                )
                time.sleep(IP_BLOCK_WAIT)
                if consecutive_ip_blocks >= MAX_IP_BLOCKS:
                    tqdm.write('  [STOPPING] Too many consecutive IP blocks. Checkpoint saved.')
                    return '', 'ip_blocked'
                continue

            if _is_unavailable(err_type):
                consecutive_ip_blocks = 0
                return '', 'unavailable'

            if attempt < RETRY_ON_ERROR:
                tqdm.write(f'  [RETRY {attempt+1}/{RETRY_ON_ERROR}] {video_id}: {err_type}: {err_msg[:80]}')
                time.sleep(RETRY_DELAY * (attempt + 1))
                continue

            consecutive_ip_blocks = 0
            tqdm.write(f'  [ERROR] {video_id}: {err_type}: {err_msg[:120]}')
            return '', 'error'

    return '', 'error'


# ── Main loop ────────────────────────────────────────────────────────────────

rows_this_run = 0

progress = tqdm(dataset, desc='Fetching captions', unit='sample')
for sample in progress:
    transcript = (sample.get('Transcript') or '').strip()
    if not transcript:
        continue

    sample_id = str(sample.get('ID', ''))

    if sample_id in completed_ids:
        continue

    if ip_block_stop:
        hyp_raw, caption_type = '', 'ip_blocked'
    else:
        hyp_raw, caption_type = fetch_caption(sample_id)
        if caption_type == 'ip_blocked' and consecutive_ip_blocks >= MAX_IP_BLOCKS:
            ip_block_stop = True

    stats[caption_type] = stats.get(caption_type, 0) + 1

    row = {
        'split':                     'test',
        'ID':                        sample_id,
        'Speaker_ID':                sample.get('Speaker_ID', ''),
        'Gender':                    sample.get('Gender', ''),
        'Speech_Class':              sample.get('Speech_Class', ''),
        'Native_Region':             sample.get('Native_Region', ''),
        'Speech_Duration_seconds':   sample.get('Speech_Duration_seconds') or '',
        'Discipline_Group':          sample.get('Discipline_Group', ''),
        'Topic':                     sample.get('Topic', ''),
        'transcript_raw':            transcript,
        'normalised_transcript_raw': str(sample.get('Normalised_Transcript') or '').strip(),
        'hypothesis_raw':            hyp_raw,
        'caption_type':              caption_type,
    }

    # Append to Drive checkpoint immediately — survives any crash
    write_header = not os.path.exists(CHECKPOINT_PATH)
    pd.DataFrame([row]).to_csv(
        CHECKPOINT_PATH,
        mode='a',
        header=write_header,
        index=False,
    )

    completed_ids.add(sample_id)
    rows_this_run += 1

    total_done = len(completed_ids)
    progress.set_postfix(
        done=total_done,
        manual=stats['manual'],
        auto=stats['auto'],
        unavail=stats['unavailable'],
        blocked=stats['ip_blocked'],
        err=stats['error'],
    )

    if rows_this_run % 50 == 0:
        tqdm.write(
            f'[{total_done}/{len(dataset)}] '
            f'manual:{stats["manual"]} auto:{stats["auto"]} '
            f'unavail:{stats["unavailable"]} blocked:{stats["ip_blocked"]} '
            f'err:{stats["error"]}'
        )

    if ip_block_stop:
        tqdm.write(
            f'\n[STOPPED] IP blocked after {MAX_IP_BLOCKS} consecutive blocks.\n'
            f'  {len(completed_ids)}/{len(dataset)} rows saved to checkpoint.\n'
            f'  Factory reset runtime (Runtime → Factory reset) to get a fresh IP,\n'
            f'  then re-run Cell 4 → Cell 5 to resume.'
        )
        break

    if not ip_block_stop:
        time.sleep(DELAY_BETWEEN_REQUESTS)

# ── Session summary ───────────────────────────────────────────────────────────
total_done = len(completed_ids)
remaining  = len(dataset) - total_done
available  = stats['manual'] + stats['auto']

print(f'\n── Session complete ──────────────────────────────────────────────')
print(f'  This run        : {rows_this_run} new rows processed')
print(f'  Total done      : {total_done}/{len(dataset)}')
print(f'  Remaining       : {remaining}')
print(f'  manual          : {stats["manual"]}')
print(f'  auto            : {stats["auto"]}')
print(f'  unavailable     : {stats["unavailable"]}')
print(f'  ip_blocked      : {stats["ip_blocked"]}')
print(f'  error           : {stats["error"]}')
if total_done:
    print(f'  Coverage        : {available}/{total_done} ({available/total_done*100:.1f}%)')

if remaining == 0:
    print('\n[ALL DONE] Run Cell 6 to write the final wer_youtube_raw.csv')
elif ip_block_stop:
    print('\n[BLOCKED] Factory reset runtime → re-run Cell 4 → Cell 5 to resume.')
else:
    print('\n[PARTIAL] Re-run Cell 4 then Cell 5 to continue.')

Fetching captions:   0%|          | 0/986 [00:00<?, ?sample/s]

[962/986] manual:211 auto:0 unavail:880 blocked:1 err:10

── Session complete ──────────────────────────────────────────────
  This run        : 74 new rows processed
  Total done      : 986/986
  Remaining       : 0
  manual          : 211
  auto            : 0
  unavailable     : 904
  ip_blocked      : 1
  error           : 10
  Coverage        : 211/986 (21.4%)

[ALL DONE] Run Cell 6 to write the final wer_youtube_raw.csv


In [6]:
# ── Cell 6: Finalise ─────────────────────────────────────────────────────────
# Run only when all rows are done (remaining == 0 above).
# Reads the checkpoint, validates schema, writes final wer_youtube_raw.csv.

import os
import pandas as pd

EXPECTED_COLUMNS = [
    'split', 'ID', 'Speaker_ID', 'Gender', 'Speech_Class', 'Native_Region',
    'Speech_Duration_seconds', 'Discipline_Group', 'Topic',
    'transcript_raw', 'normalised_transcript_raw', 'hypothesis_raw', 'caption_type',
]

# ── Load checkpoint ──────────────────────────────────────────────────────────
if not os.path.exists(CHECKPOINT_PATH):
    raise FileNotFoundError(f'Checkpoint not found: {CHECKPOINT_PATH}\nRun Cell 5 first.')

df = pd.read_csv(CHECKPOINT_PATH)
print(f'Checkpoint loaded: {len(df)} rows')

# ── Validate schema ──────────────────────────────────────────────────────────
missing_cols = [c for c in EXPECTED_COLUMNS if c not in df.columns]
if missing_cols:
    raise ValueError(f'Checkpoint missing columns: {missing_cols}')

# Remove any duplicate IDs (keep last — most recent fetch)
before = len(df)
df = df.drop_duplicates(subset='ID', keep='last').reset_index(drop=True)
if len(df) < before:
    print(f'  Removed {before - len(df)} duplicate IDs.')

# Enforce column order
df = df[EXPECTED_COLUMNS]

# ── Coverage check ───────────────────────────────────────────────────────────
total      = len(df)
available  = (df['caption_type'].isin(['manual', 'auto'])).sum()
unavail    = (df['caption_type'] == 'unavailable').sum()
blocked    = (df['caption_type'] == 'ip_blocked').sum()
errors     = (df['caption_type'] == 'error').sum()
manual_ct  = (df['caption_type'] == 'manual').sum()
auto_ct    = (df['caption_type'] == 'auto').sum()

print(f'\n── Final statistics ─────────────────────────────────────────────')
print(f'  Total rows      : {total}')
print(f'  manual          : {manual_ct}')
print(f'  auto            : {auto_ct}')
print(f'  unavailable     : {unavail}')
print(f'  ip_blocked      : {blocked}')
print(f'  error           : {errors}')
print(f'  Coverage        : {available}/{total} ({available/total*100:.1f}%)' if total else '')

if blocked > 0:
    print(f'\n  WARNING: {blocked} rows are ip_blocked — captions not fetched.')
    print('  Re-run Cell 4 + Cell 5 from a different network to fetch them.')
    print('  Proceeding to save partial results anyway.')

# ── Write final CSV ──────────────────────────────────────────────────────────
df.to_csv(FINAL_PATH, index=False)
print(f'\nSaved: {FINAL_PATH}')

print("""
── Next steps ────────────────────────────────────────────────────────────────
1. Open Google Drive → indian-asr-bench/
2. Download wer_youtube_raw.csv
3. Place it at:
     results/stage1_raw_transcripts/wer_youtube_raw.csv
4. Run from the repo root:
     python normalize_and_score.py
     python analysis/compare_all.py
─────────────────────────────────────────────────────────────────────────────
""")

Checkpoint loaded: 1126 rows
  Removed 140 duplicate IDs.

── Final statistics ─────────────────────────────────────────────
  Total rows      : 986
  manual          : 190
  auto            : 0
  unavailable     : 796
  ip_blocked      : 0
  error           : 0
  Coverage        : 190/986 (19.3%)

Saved: /content/drive/MyDrive/indian-asr-bench/wer_youtube_raw.csv

── Next steps ────────────────────────────────────────────────────────────────
1. Open Google Drive → indian-asr-bench/
2. Download wer_youtube_raw.csv
3. Place it at:
     results/stage1_raw_transcripts/wer_youtube_raw.csv
4. Run from the repo root:
     python normalize_and_score.py
     python analysis/compare_all.py
─────────────────────────────────────────────────────────────────────────────

